# Mini-Project MP03 — Press Release to Plot

## Industry Comparison: Financial Services and Travel and Hospitality

*CIS 3120 — Programming for Analytics*
*Baruch College, Zicklin School of Business*

---

**Team number:** `<NN>` (replace with two-digit number from Brightspace)

**Team members:**
- Financial Services Pipeline Lead: `<name>`
- Travel and Hospitality Pipeline Lead: `<name>`
- Comparison and Visualization Lead (Integrator): `<name>`

**Submission filename:** `MP03_Notebook_team_<NN>.ipynb`

---

## How to use this starter

1. Make a copy of this notebook and rename it `MP03_Notebook_team_<NN>.ipynb` using your team number.
2. Replace the User-Agent placeholder in the setup cell with your Baruch email.
3. Configure your Anthropic API key in Colab Secrets as `ANTHROPIC_API_KEY`.
4. Work through the notebook section by section. Sections marked **CANONICAL** are the validated Module 15 pipeline and must not be modified. Sections marked **TODO** are where your team writes new code.
5. Run the window-tuning experiment, populate the results table, build the integrated map, and complete the methodology and reflection sections.
6. Verify the notebook runs end-to-end (Runtime → Restart and run all in Colab) before submitting.

See `docs/MP03_Assignment.docx` for the full assignment specification.

---

## 1. Setup

Install dependencies (Colab) and configure the request headers and API client.

In [31]:
# Colab installs (silent). The other packages are pre-installed in the Colab base image.
!pip install anthropic folium --quiet

In [21]:
import json
import re
import time
from datetime import date, datetime, timedelta

import requests
from bs4 import BeautifulSoup
import folium
import pandas as pd
from anthropic import Anthropic

# ─────────────────────────────────────────────────────────────────────────
# CRITICAL: Replace the placeholder below with your Baruch email.
# Both SEC EDGAR and OpenStreetMap Nominatim require a descriptive
# User-Agent header. Generic agents are rejected with HTTP 403.
# ─────────────────────────────────────────────────────────────────────────
USER_AGENT = "CIS3120 MP03 Team <NN> - your.name@baruch.cuny.edu"

REQUEST_HEADERS = {"User-Agent": USER_AGENT}

# ─────────────────────────────────────────────────────────────────────────
# Endpoints and constants
# ─────────────────────────────────────────────────────────────────────────
EDGAR_SEARCH_URL = "https://efts.sec.gov/LATEST/search-index"
NOMINATIM_URL    = "https://nominatim.openstreetmap.org/search"

EDGAR_PAUSE      = 0.15   # seconds between EDGAR requests (SEC: 10 req/sec)
NOMINATIM_PAUSE  = 1.10   # seconds between Nominatim requests (1 req/sec)

# Anthropic model: current Haiku in the Claude 4.5 family.
MODEL_ID = "claude-haiku-4-5-20251001"

In [32]:
from google.colab import userdata

ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
client = Anthropic(api_key=ANTHROPIC_API_KEY)

In [33]:
import sys
sys.path.insert(0, "/content")

In [34]:
import os

# Content of /mp03/__init__.py
init_py_content = """
"""

# Content of /mp03/seeds.py
seeds_py_content = """
FINANCIAL_SERVICES_TICKERS = [
    "JPM", # JPMorgan Chase & Co.
    "BAC", # Bank of America Corp
    "WFC", # Wells Fargo & Co
    "C",   # Citigroup Inc
    "V",   # Visa Inc.
    "MA",  # Mastercard Inc.
    "PYPL", # PayPal Holdings Inc
    "GS",  # Goldman Sachs Group Inc
    "MS",  # Morgan Stanley
    "BLK", # BlackRock Inc
    "SPGI", # S&P Global Inc
    "ICE", # Intercontinental Exchange Inc
    "CME", # CME Group Inc
    "MSCI", # MSCI Inc
    "AXP", # American Express Co
    "DFS", # Discover Financial Services
    "COF", # Capital One Financial Corp
    "PNC", # PNC Financial Services Group Inc
    "USB", # U.S. Bancorp
    "SCHW", # Charles Schwab Corp
    "GSBD", # Goldman Sachs BDC, Inc. - example of a BDC for diversification
    "HTGC", # Hercules Capital, Inc. - another BDC example
    "BX",  # Blackstone Inc. - asset management
    "KKR", # KKR & Co. Inc. - private equity
    "ARES", # Ares Management Corporation - alternative asset manager
    "RJF", # Raymond James Financial, Inc. - diversified financial services
    "LPLA", # LPL Financial Holdings Inc. - broker-dealer, investment advisor
    "NDAQ", # Nasdaq, Inc. - exchange operator
    "MKTX", # MarketAxess Holdings Inc. - electronic trading platform
]

FINANCIAL_SERVICES_PHRASES = [
    # Openings
    '"branch opening"',
    '"new office"',
    '"launch of a new location"',
    '"grand opening"',
    '"store opening"',
    '"inauguration of"',
    '"new facility"',
    '"opening of its first"',
    '"new headquarters"',
    '"expansion into"',
    '"expanded its presence"',
    '"introduces new banking center"',
    '"opens new financial center"',
    '"establishes new advisory hub"',

    # Closings
    '"branch closure"',
    '"office closure"',
    '"closing of its"',
    '"ceasing operations at"',
    '"consolidating its offices"',
    '"shutting down operations"',
    '"discontinuation of services at"',
    '"divesting its operations in"',
    '"reduces physical footprint"',
    '"merger of branches"',

    # Relocations
    '"relocated its office"',
    '"moving its headquarters"',
    '"new location for its operations"',
    '"transfer of its facilities"',
    '"moving to a new building"',
    '"relocation of its main branch"',
    '"consolidates its offices to a new location"',

    # Expansions
    '"expanding its presence"',
    '"expansion of its"',
    '"increased capacity at"',
    '"adding new space"',
    '"renovating its facilities"',
    '"upgrading its infrastructure at"',
    '"doubling its footprint"',
    '"larger facility in"',
    '"growing its operations in"',
    '"significant investment in its facility"',
    '"adding new data center capacity"',
    '"new innovation lab"',
]

TRAVEL_HOSPITALITY_TICKERS = [
    "MAR", # Marriott International, Inc.
    "HLT", # Hilton Worldwide Holdings Inc.
    "H",   # Hyatt Hotels Corporation
    "MGM", # MGM Resorts International
    "LVS", # Las Vegas Sands Corp.
    "WYNN", # Wynn Resorts, Limited
    "RCL", # Royal Caribbean Group
    "CCL", # Carnival Corporation & plc
    "NCLH", # Norwegian Cruise Line Holdings Ltd.
    "BKNG", # Booking Holdings Inc.
    "EXPE", # Expedia Group, Inc.
    "ABNB", # Airbnb, Inc.
    "DHR", # Danaher Corporation (diversified, but has some travel-related segments like life sciences tools)
    "TRIP", # Tripadvisor, Inc.
    "DAL", # Delta Air Lines, Inc.
    "UAL", # United Airlines Holdings, Inc.
    "AAL", # American Airlines Group Inc.
    "LUV", # Southwest Airlines Co.
    "CZR", # Caesars Entertainment, Inc.
    "SIX", # Six Flags Entertainment Corporation
    "EPR", # EPR Properties (REIT focused on entertainment, recreation and leisure)
    "PLYA", # Playa Hotels & Resorts N.V.
    "DRH", # Diamondrock Hospitality Company
    "PEB", # Pebblebrook Hotel Trust
]

TRAVEL_HOSPITALITY_PHRASES = [
    # Openings
    '"hotel opening"',
    '"resort opening"',
    '"new restaurant"',
    '"grand opening"',
    '"new location"',
    '"launch of"',
    '"opening of a new"',
    '"inauguration of"',
    '"debuts new property"',
    '"expansion into"',
    '"starts operations in"',
    '"unveils new attraction"',

    # Closings
    '"hotel closure"',
    '"resort closure"',
    '"restaurant closure"',
    '"closing of its"',
    '"ceasing operations at"',
    '"shuttering of"',
    '"discontinuation of services at"',
    '"selling its property in"',
    '"divesting its assets in"',
    '"phasing out operations at"',

    # Relocations
    '"relocated its office"',
    '"moving its headquarters"',
    '"new location for its operations"',
    '"transfer of its facilities"',
    '"moving to a new building"',
    '"relocation of its main booking office"',
    '"consolidates its call center to a new location"',

    # Expansions
    '"expanding its capacity"',
    '"expansion of its"',
    '"increased capacity at"',
    '"adding new space"',
    '"renovating its facilities"',
    '"upgrading its infrastructure at"',
    '"adding new rooms"',
    '"new wing"',
    '"doubling its footprint"',
    '"larger facility in"',
    '"growing its operations in"',
    '"significant investment in its property"',
    '"new entertainment complex"',
]
"""

# Write __init__.py
with open("/content/mp03/__init__.py", "w") as f:
    f.write(init_py_content)

# Write seeds.py
with open("/content/mp03/seeds.py", "w") as f:
    f.write(seeds_py_content)

print("mp03/__init__.py and mp03/seeds.py created successfully in /content/mp03/")
!ls /content/mp03

mp03/__init__.py and mp03/seeds.py created successfully in /content/mp03/
__init__.py  seeds.py


In [35]:
# Re-run the import cell after creating the files
# If the mp03 package is not on the Python path, append the parent directory.
import sys
from pathlib import Path

# When running in Colab from a cloned repo, this places the repo root on sys.path.
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from mp03.seeds import (
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
)

print(f"Financial Services tickers: {len(FINANCIAL_SERVICES_TICKERS)}")
print(f"Financial Services phrases: {len(FINANCIAL_SERVICES_PHRASES)}")
print(f"Travel and Hospitality tickers: {len(TRAVEL_HOSPITALITY_TICKERS)}")
print(f"Travel and Hospitality phrases: {len(TRAVEL_HOSPITALITY_PHRASES)}")

Financial Services tickers: 29
Financial Services phrases: 43
Travel and Hospitality tickers: 24
Travel and Hospitality phrases: 42


In [36]:
# Import the seeded ticker lists and search-phrase lists from the mp03 module.
# If the mp03 package is not on the Python path, append the parent directory.
import sys
from pathlib import Path

# When running in Colab from a cloned repo, this places the repo root on sys.path.
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from mp03.seeds import (
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
)

print(f"Financial Services tickers: {len(FINANCIAL_SERVICES_TICKERS)}")
print(f"Financial Services phrases: {len(FINANCIAL_SERVICES_PHRASES)}")
print(f"Travel and Hospitality tickers: {len(TRAVEL_HOSPITALITY_TICKERS)}")
print(f"Travel and Hospitality phrases: {len(TRAVEL_HOSPITALITY_PHRASES)}")

Financial Services tickers: 29
Financial Services phrases: 43
Travel and Hospitality tickers: 24
Travel and Hospitality phrases: 42


---

## 2. Canonical Pipeline (Module 15)

The five functions in this section are the preserved pipeline from the Module 15 instructor notebook. **Do not modify these signatures.** Downstream code in this notebook calls them with these exact argument shapes.

### Stage 1 — Retrieve candidate 8-K filings from EDGAR

Each phrase is queried independently. Combining phrases with boolean OR inside parentheses is a documented but non-functional approach in the SEC's full-text search engine and must not be used.

In [37]:
def search_edgar_one_phrase(
    phrase: str,
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
) -> tuple[list[dict], int]:
    """Query EDGAR full-text search for one phrase across a date window.

    Returns a tuple of (list of hit dicts, total reported by EDGAR).
    """
    all_hits: list[dict] = []
    total = 0
    for page in range(max_pages):
        params = {
            "q":         phrase,
            "dateRange": "custom",
            "startdt":   start_date.isoformat(),
            "enddt":     end_date.isoformat(),
            "forms":     forms,
            "from":      page * 100,
        }
        response = requests.get(
            EDGAR_SEARCH_URL,
            params=params,
            headers=REQUEST_HEADERS,
            timeout=30,
        )
        response.raise_for_status()
        data = response.json()
        hits = data.get("hits", {}).get("hits", [])
        all_hits.extend(hits)
        total = data.get("hits", {}).get("total", {}).get("value", 0)
        if (page + 1) * 100 >= total:
            break
        time.sleep(EDGAR_PAUSE)
    return all_hits, total

In [38]:
def search_edgar_all_phrases(
    phrases: list[str],
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
    max_filings: int = 250,
) -> list[dict]:
    """Run search_edgar_one_phrase across a list of phrases with retry-with-backoff.

    Deduplicates by (accession number, exhibit filename). Stops accumulating
    once max_filings is reached.
    """
    seen: set[str] = set()
    deduped: list[dict] = []
    backoff_waits = [5, 10, 15]

    for phrase in phrases:
        attempts = 0
        while attempts <= len(backoff_waits):
            try:
                hits, _ = search_edgar_one_phrase(
                    phrase, start_date, end_date, forms, max_pages
                )
                break
            except requests.RequestException as exc:
                if attempts == len(backoff_waits):
                    print(f"  WARNING: phrase {phrase!r} failed after retries ({exc}); skipping")
                    hits = []
                    break
                wait = backoff_waits[attempts]
                print(f"  transient error on {phrase!r}: {exc}. retrying in {wait}s...")
                time.sleep(wait)
                attempts += 1

        for hit in hits:
            key = hit.get("_id", "")
            if key and key not in seen:
                seen.add(key)
                deduped.append(hit)
            if len(deduped) >= max_filings:
                return deduped
        time.sleep(EDGAR_PAUSE)

    return deduped

### Stage 2 — Fetch the press release text from each filing

In [39]:
def build_exhibit_url(hit: dict) -> str:
    """Construct the SEC archive URL for the exhibit referenced by the hit."""
    accession_full, filename = hit["_id"].split(":")
    accession_no_dashes = accession_full.replace("-", "")
    cik = hit["_source"]["ciks"][0].lstrip("0")
    return (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{cik}/{accession_no_dashes}/{filename}"
    )


def fetch_exhibit_text(hit: dict, max_chars: int = 8000) -> tuple[str, str]:
    """Fetch and HTML-strip the exhibit text for a single hit.

    Returns (text, url). Truncates at max_chars (~2000 tokens).
    """
    url = build_exhibit_url(hit)
    response = requests.get(url, headers=REQUEST_HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    if len(text) > max_chars:
        text = text[:max_chars] + " […truncated…]"
    return text, url

### Stage 3 — Classify and extract with the Anthropic API

The system prompt below achieved 100 percent precision in prototype testing. Use it verbatim.

In [40]:
EXTRACTION_SYSTEM_PROMPT = """You are an analyst reviewing 8-K filing exhibits to identify announcements of location-related corporate events: openings, closings, relocations, or expansions of physical facilities (stores, warehouses, distribution centers, offices, plants).

Return ONLY a JSON object with these exact fields:
- is_location_event: boolean. True ONLY if the filing genuinely announces opening, closing, relocation, or expansion of a specific physical facility at a named location. False for earnings, executive changes, financing, share repurchases, generic corporate updates, or mentions of locations that are not the subject of the announcement.
- event_type: one of "opening", "closing", "relocation", "expansion", "other", or null
- city: string with the city name, or null if no specific city is named
- state: two-letter US state code (e.g., "NY", "CA"), or null if not US-based or not specified
- summary: one sentence (under 25 words) describing the event in plain language, or null

Be strict. If the filing mentions a location only in passing (e.g., headquarters address in the boilerplate), return is_location_event: false. Return only the JSON object with no preamble, no markdown fences, no explanation."""


def extract_with_claude(filing: dict) -> dict:
    """Classify and extract structured location data from a single filing.

    Expects filing dict with keys: text (str), url (str), and any other
    metadata to be preserved on the returned record. Returns a dict
    extending filing with the parsed extraction fields and token usage.
    """
    response = client.messages.create(
        model=MODEL_ID,
        max_tokens=300,
        system=EXTRACTION_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": filing["text"]}],
    )

    raw = response.content[0].text.strip()
    raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        parsed = {"is_location_event": False, "_parse_error": raw[:200]}

    record = {**filing, **parsed}
    record["input_tokens"]  = response.usage.input_tokens
    record["output_tokens"] = response.usage.output_tokens
    return record

### Stage 4 — Geocode the locations

Nominatim enforces a strict 1-request-per-second policy. The 1.10-second pause is a comfortable margin.

In [41]:
def geocode_location(city: str, state: str | None) -> tuple[float, float] | None:
    """Geocode a US city/state pair via OpenStreetMap Nominatim.

    Returns (latitude, longitude) on success, None if no match is found.
    """
    if not city:
        return None
    query = f"{city}, {state}, USA" if state else f"{city}, USA"
    params = {"q": query, "format": "json", "limit": 1, "countrycodes": "us"}
    response = requests.get(
        NOMINATIM_URL,
        params=params,
        headers=REQUEST_HEADERS,
        timeout=30,
    )
    response.raise_for_status()
    data = response.json()
    time.sleep(NOMINATIM_PAUSE)
    if not data:
        return None
    return float(data[0]["lat"]), float(data[0]["lon"])

### Stage 5 — Render the folium map (base configuration)

The base map and event color palette are provided. Your team will customize the marker rendering in Section 5 below to encode both industry and event type.

In [42]:
EVENT_COLORS = {
    "opening":    "green",
    "closing":    "red",
    "relocation": "orange",
    "expansion":  "blue",
    "other":      "gray",
}

# Reasonable default center (geographic center of the contiguous US).
US_CENTER_LAT = 39.8
US_CENTER_LON = -98.6

---

## 3. Required New Functions (TODO)

Each team adds the three functions below. Each one has a single, well-defined responsibility. Do not bundle multiple responsibilities into one function.

Reference: `docs/MP03_Assignment.docx`, Section 3.

In [43]:
def filter_candidates_by_tickers(
    candidates: list[dict],
    ticker_list: list[str],
) -> list[dict]:
    """Restrict a candidate set returned by Stage 1 to a list of tickers."""

    ticker_set = {ticker.upper() for ticker in ticker_list}

    filtered = []

    for hit in candidates:
        hit_tickers = hit.get("_source", {}).get("tickers", [])

        hit_tickers_upper = {ticker.upper() for ticker in hit_tickers}

        if ticker_set.intersection(hit_tickers_upper):
            filtered.append(hit)

    return filtered

In [44]:
def run_industry_pipeline(
    industry_label: str,
    ticker_list: list[str],
    phrase_list: list[str],
    window_days: int,
) -> list[dict]:
    """Run all five pipeline stages for one industry slice."""

    end_date = date.today()
    start_date = end_date - timedelta(days=window_days)

    candidates = search_edgar_all_phrases(
        phrase_list,
        start_date,
        end_date,
    )

    filtered_candidates = filter_candidates_by_tickers(
        candidates,
        ticker_list,
    )

    geocoded_events = []

    for candidate in filtered_candidates:
        exhibit_url = build_exhibit_url(candidate)
        exhibit_text, final_url = fetch_exhibit_text(candidate)

        filing = {
            "hit": candidate,
            "exhibit_url": exhibit_url,
            "final_url": final_url,
            "text": exhibit_text,
        }

        extracted = extract_with_claude(filing)

        if not extracted.get("is_location_event"):
            continue

        city = extracted.get("city")
        state = extracted.get("state")

        if not city:
            continue

        coordinates = geocode_location(city, state)

        if coordinates is None:
            continue

        latitude, longitude = coordinates

        extracted["latitude"] = latitude
        extracted["longitude"] = longitude
        extracted["industry"] = industry_label
        extracted["exhibit_url"] = final_url

        geocoded_events.append(extracted)

    return geocoded_events

In [45]:
def summarize_window_trial(
    industry_label: str,
    window_days: int,
    candidate_count: int,
    event_count: int,
    estimated_cost_usd: float,
) -> dict:
    """Record the result of one window-tuning trial."""

    return {
        "industry": industry_label,
        "window_days": window_days,
        "candidate_count": candidate_count,
        "event_count": event_count,
        "estimated_cost_usd": estimated_cost_usd,
    }

---

## 4. Window-Tuning Experiment

Determine the smallest window that produces at least 8 location events for both industries without exceeding the $3.00 cumulative cost ceiling.

**Protocol:**
1. Begin at `WINDOW_DAYS = 30`. Run the pipeline for both industries.
2. If both industries reach the event-count target, stop.
3. Otherwise advance through 60, 90, 180, 360. Stop at the first window where both industries reach the target, or at 360, whichever comes first.

**Stopping criteria:**

| Criterion | Threshold |
|:---|:---|
| Event-count target | At least 8 location events per industry |
| Cost ceiling | $3.00 cumulative across all trials |
| Window ceiling | 360 days |

Reference: `docs/MP03_Assignment.docx`, Section 4.

In [46]:
# Initialize the window-experiment results table.
# Append one row per (industry, window) trial that you actually run.
window_results = pd.DataFrame(columns=[
    "industry",
    "window_days",
    "candidate_count",
    "event_count",
    "estimated_cost_usd",
])

window_results

,industry,window_days,candidate_count,event_count,estimated_cost_usd


### 4.1 Window trials — Financial Services

Run the pipeline for Financial Services at successive window lengths and append a row to `window_results` after each trial using `summarize_window_trial`.

In [47]:
travel_events_30 = run_industry_pipeline(
    "Travel and Hospitality",
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
    window_days=30,
)

travel_row_30 = summarize_window_trial(
    industry_label="Travel and Hospitality",
    window_days=30,
    candidate_count=len(travel_events_30),
    event_count=len(travel_events_30),
    estimated_cost_usd=0.25,
)

window_results = pd.concat(
    [window_results, pd.DataFrame([travel_row_30])],
    ignore_index=True,
)

window_results

/tmp/ipykernel_1279/2943437438.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  window_results = pd.concat(


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Travel and Hospitality,30,0,0,0.25


In [48]:
travel_events_90 = run_industry_pipeline(
    "Travel and Hospitality",
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
    window_days=90,
)

travel_row_90 = summarize_window_trial(
    industry_label="Travel and Hospitality",
    window_days=90,
    candidate_count=len(travel_events_90),
    event_count=len(travel_events_90),
    estimated_cost_usd=0.75,
)

window_results = pd.concat(
    [window_results, pd.DataFrame([travel_row_90])],
    ignore_index=True,
)

window_results

,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Travel and Hospitality,30,0,0,0.25
1,Travel and Hospitality,90,0,0,0.75


In [49]:
travel_events_360 = run_industry_pipeline(
    "Travel and Hospitality",
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
    window_days=360,
)

travel_row_360 = summarize_window_trial(
    industry_label="Travel and Hospitality",
    window_days=360,
    candidate_count=len(travel_events_360),
    event_count=len(travel_events_360),
    estimated_cost_usd=2.00,
)

window_results = pd.concat(
    [window_results, pd.DataFrame([travel_row_360])],
    ignore_index=True,
)

window_results

,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Travel and Hospitality,30,0,0,0.25
1,Travel and Hospitality,90,0,0,0.75
2,Travel and Hospitality,360,0,0,2.00


### 4.2 Window trials — Travel and Hospitality

In [ ]:
# TODO: run window trials for Travel and Hospitality (analogous to 4.1 above).

### 4.3 Selected window and final pipeline runs

Once both industries reach the event-count target at a common window length, record the chosen window below and run the final pipeline for both industries at that window. The events from these two final runs feed Section 5.

In [50]:
CHOSEN_WINDOW_DAYS = 360

th_events = run_industry_pipeline(
    "Travel and Hospitality",
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
    window_days=CHOSEN_WINDOW_DAYS,
)

all_events = th_events

print(f"Travel and Hospitality: {len(th_events)} events")
print(f"Total:                  {len(all_events)} events")

Travel and Hospitality: 0 events
Total:                  0 events


---

## 5. Integrated Folium Map

Build a single map containing markers from both industries. The visual encoding must distinguish industry and event type **simultaneously and unambiguously**. The recommended scheme is:

- **Industry** by marker color family (e.g., navy for Financial Services, teal for Travel and Hospitality).
- **Event type** by marker icon shape (e.g., `home` for opening, `times-circle` for closing).

Each marker's popup must display: company name, ticker, industry label, filing date, event type, summary, and a working hyperlink to the underlying SEC filing.

Reference: `docs/MP03_Assignment.docx`, Section 7 (verification checklist).

In [51]:
US_CENTER_LAT = 39.8283
US_CENTER_LON = -98.5795

industry_colors = {
    "Financial Services": "darkblue",
    "Travel and Hospitality": "cadetblue",
}

event_icons = {
    "opening": "home",
    "closing": "times-circle",
    "relocation": "exchange-alt",
    "expansion": "plus-circle",
}

m = folium.Map(
    location=[US_CENTER_LAT, US_CENTER_LON],
    zoom_start=4,
    tiles="CartoDB positron",
)

for event in all_events:
    event_type = event.get("event_type", "other")
    industry = event.get("industry", "Unknown")

    popup_html = f"""
    <b>Company:</b> {event.get("company_name", "Unknown")}<br>
    <b>Ticker:</b> {event.get("ticker", "Unknown")}<br>
    <b>Industry:</b> {industry}<br>
    <b>Filing Date:</b> {event.get("filing_date", "Unknown")}<br>
    <b>Event Type:</b> {event_type}<br>
    <b>Summary:</b> {event.get("summary", "No summary available")}<br>
    <a href="{event.get("exhibit_url", "#")}" target="_blank">SEC filing</a>
    """

    latitude = event.get("lat", event.get("latitude"))
    longitude = event.get("lon", event.get("longitude"))

    if latitude is None or longitude is None:
        continue

    marker = folium.Marker(
        location=[latitude, longitude],
        popup=folium.Popup(popup_html, max_width=350),
        icon=folium.Icon(
            color=industry_colors.get(industry, "gray"),
            icon=event_icons.get(event_type, "info-circle"),
            prefix="fa",
        ),
    )
    marker.add_to(m)

m

### Export the map to `maps/mp03_map_team_<NN>.html`

In [52]:
OUTPUT_PATH = "/content/mp03_map_team_22.html"

m.save(OUTPUT_PATH)

print(f"Map saved to {OUTPUT_PATH}")

Map saved to /content/mp03_map_team_22.html


---

## 6. Methodology

The content below also appears as a standalone Markdown file at `methodology/mp03_methodology_team_<NN>.md`. Both copies must contain the same content; the standalone file is the version graded.

### 6.1 Ticker-list rationale

*TODO: For each industry, justify any modifications to the seeded ticker list. Identify what the seeded list undercounts or overcounts and explain how your changes address those limitations.*

### 6.2 Search-phrase rationale

*TODO: For each industry, justify any modifications to the seeded phrase list. Note any phrases that returned high-volume false positives or missed event categories the team considered important.*

### 6.3 Window-experiment results

*TODO: Insert the populated `window_results` table here (as Markdown) and explain why the chosen window is appropriate. Address the cost ceiling explicitly.*

### 6.4 Stage 3 classification quality per industry

*TODO: For each industry, document observed precision and any patterns in the Stage 3 classifications (false positives, false negatives, ambiguous cases). Use small numerical examples where possible.*

### 6.5 Limitations

*TODO: Identify limitations the team encountered and discuss how each affects the comparative reflection.*

---

## 7. Comparative Reflection

A 300-to-400-word reflection on what the geographic patterns reveal about how the two industries deploy and consolidate physical capacity, and what the differences imply about each industry's underlying economics.

The same content appears as a standalone Markdown file at `reflections/mp03_reflection_team_<NN>.md`.

*TODO: Write the comparative reflection here. Mere description of the maps does not earn full credit; the reflection must offer substantive interpretation grounded in the underlying business economics and address limitations honestly.*

---

## 8. Pre-Submission Verification

Before the integrator submits, confirm each of the following:

- [ ] Notebook restarts cleanly and runs end-to-end (Runtime → Restart and run all in Colab).
- [ ] No committed API keys, no hard-coded credentials, no leftover debug prints.
- [ ] `window_results` table is populated with at least one row per (industry, window) trial actually run.
- [ ] Both industries reach at least 8 location events at the chosen window, OR a 360-day trial was run for both and the short-fall is acknowledged in Section 6.
- [ ] Cumulative window-tuning cost is at or below $3.00.
- [ ] Integrated map renders inline AND is exported to `maps/mp03_map_team_<NN>.html`.
- [ ] Every marker has a popup with all required fields and a working SEC hyperlink.
- [ ] Industry is visually distinguishable from event type on the map.
- [ ] Methodology appears both in this notebook and at `methodology/mp03_methodology_team_<NN>.md`.
- [ ] Comparative reflection appears both in this notebook and at `reflections/mp03_reflection_team_<NN>.md`.
- [ ] Team branch name is exactly `mp/03-industry-comparison-team-<NN>` and submission tag `mp03-team-<NN>` is pushed.
- [ ] At least three commits per team member following the `feat(scope): description` convention appear in the merged history.
- [ ] Brightspace submission text field contains the upstream PR URL and the names of all three team members with their roles.